## Implementacion del algortimo Probabilistico CLR-2 

-  Autor: Germán Homero Morán Figueroa
- Descripción: En este notebook se realiza la implmentación del algortimo probabilistico, variando la semilla (s) y el numero de grupos (k). En este caso se prueban con 3 algoritmos de clustering definidos en la literatura para realizar la asignación inicial de las observaciones
    - Mezclas Gausinas (GMM)
    - k- Means 
    - Spectral Clustering

salida: Artefactos del modelo (archivos .pkl) para regresión y clasificación.

> Notas
* Para definir el numero de grupos (k), en este caso se establece en 2, dado que al variar el numero de grupos mediante un ciclo for se obtuvo que los mejroes resultados fueron con k=2



In [1]:
# ==============================================================================
import pandas as pd
import numpy as np

# Gráficos
# ==============================================================================
import matplotlib.pyplot as plt
from matplotlib import style
import seaborn as sns

# Preprocesado y modelado
# ==============================================================================
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error,accuracy_score,mean_absolute_error
import statsmodels.api as sm
import statsmodels.formula.api as smf
import xgboost as xgb
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingRegressor
from imblearn.over_sampling import SMOTE,SVMSMOTE
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.cluster import SpectralClustering
from sklearn.tree import DecisionTreeClassifier
# ======================================================================================
import joblib
from sklearn import metrics
from sklearn.cluster import SpectralClustering, KMeans, DBSCAN, AgglomerativeClustering
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import ElasticNet
import statistics
import numpy as np
from sklearn.metrics import pairwise_distances_argmin_min
from scipy.spatial.distance import cdist
import joblib



C:\Users\germanm\AppData\Roaming\Python\Python39\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\germanm\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [4]:
'''
    Esta clase permite realizar la codificación dummy especificameente variables categoricas
'''
class OneHotCoding():
    def __init__(self, df, bin_features):
        self.bin_features = bin_features
        self.df = df

    def dummyCodification(self):
        cat_features = self.df.select_dtypes(include = ["object", "category"]).columns
        bin_dataset = self.df[self.bin_features].replace({'SI': 1, 'NO': 0})
        categorical_features = [x for x in cat_features if x not in self.bin_features]
        df_cat = pd.get_dummies(self.df[categorical_features])
        self.df.drop(cat_features, axis = 1, inplace = True)
        df_final = pd.concat([self.df,df_cat,bin_dataset ], axis = 1)
        df_final.to_csv("daset_codificado.csv")
        print("Ejeción Terminada")
        return df_final

'''
    Esta clase permite Calcular modelo de regresión Elasticnet por cada grupo
'''
class LinearRegession():
    def __init__(self, df, alpha, l1_ratio):
        self.df = df
        self.alpha = alpha
        self.l1_ratio = l1_ratio


    def CalcularModeloLR(self):
        # alpha=0.1, l1_ratio=0.97
        Y = self.df.RDT_AJUSTADO.values
        X = self.df.drop(["RDT_AJUSTADO","Grupo"], axis=1).values
        modelElasticNet = ElasticNet(alpha=self.alpha, l1_ratio=self.l1_ratio, random_state=123)
        model = modelElasticNet.fit(X,Y)
        #r_2 = model.score(X,Y)
        # Pedicciones
        yhat = model.predict(X)
        r_2 = r2_score(Y, yhat)


        return [r_2,model]

'''
  Esta función permite evaluar el desempeño de los modelos de regresión
  en terminos de RMSE, MAE y R2.
'''
def metricasModelosRegresion(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    # También puedes crear un DataFrame para mostrar las métricas juntas
    metrics_df = pd.DataFrame({
        'Métrica': ['MSE', 'RMSE', 'MAE', 'R²'],
        'Valor': [mse, rmse, mae, r2]
    })
    return r2, metrics_df



'''
  Esta Función permite entrenar una Regresion lineal , teniendo en cuenta
  libreria de  Stasts models de python
  
'''

def LinearRegessionOLMS(df_entrenamiento):
    Y = df_entrenamiento.RDT_AJUSTADO.values
    X = df_entrenamiento.drop(["RDT_AJUSTADO","Grupo"], axis=1).values
    # Agreago Constantes al modelo
    X_train = sm.add_constant(X, prepend=True)
    model = sm.OLS(Y, X_train)
    res= model.fit_regularized(method='elastic_net',alpha=0.1, L1_wt=0.97)
    model_fit_regularized = model.fit(params=res.params)
    r_2 = model_fit_regularized.rsquared
    #print("Nuevo r_2: ", r_2)
    return r_2, model_fit_regularized

'''
  Función para calcular correlaciones iniciales para cada uno de los grupos
  generados por el algoritmo de clustering.
'''
def CalcularCorrelationInitialOLS(gruposDefinitivos):
    correlacionesIniciales=[]
    lista_modelos = []
    for i in range(len(gruposDefinitivos)):
        #r_2, model = LinearRegessionOLMS(gruposDefinitivos[i])
        r_2, model = LinearRegession(gruposDefinitivos[i],alpha=0.1,l1_ratio=0.97).CalcularModeloLR()
        # Guardo Correlacion
        correlacionesIniciales.append(r_2)
        lista_modelos.append(model)
        # Gurado modelos
        #name_model = f'models/modelo_ols_{i}.pkl'
        #joblib.dump(model,name_model) # Guardo el modelo.
    return correlacionesIniciales,lista_modelos




'''
  Función para dividir c/d grupo en un unico dataset
'''
def calcularGruposDefinitivos(dataset):
    ListagruposDefinitivos=[]
    for i in range(len(dataset.Grupo.unique())):
        filtro_grupo = dataset[dataset.Grupo==i]
        ListagruposDefinitivos.append(filtro_grupo)

    return ListagruposDefinitivos


'''
  Funcion para calcular el rendimiento predicho con los modelos de regresión
  entrenados para cada uno de los grupos generados por el algoritmo de clustering.
'''

def predictionYield(x_dataset_test,lista_modelos,cluster_asignado):
        y_pred_list = []
        for z in range(len(x_dataset_test)):
            #print(f"Registro {z}, cluster asignado {cluster_asignado[z]}")
            y_pred = lista_modelos[cluster_asignado[z]].predict(x_dataset_test.values[z].reshape(1,-1))
            y_pred_list.append(y_pred[0])
        return y_pred_list


'''
   - Función para aplicar algoritmos de clustering a los datos de entrenamiento.
'''
def AlgortimoClustering(dataset_train_cluster, alg, semilla):
  if alg==1:
    gm = GaussianMixture(n_components=2,covariance_type = 'full', random_state=semilla).fit(dataset_train_cluster)
    lables =  gm.predict(dataset_train_cluster)
  if alg==2:
    sc = SpectralClustering(n_clusters=2,assign_labels='discretize',random_state=semilla).fit(dataset_train_cluster)
    lables = sc.labels_
  if alg==3:
    km = KMeans(n_clusters=2, random_state=semilla, n_init=10).fit(dataset_train_cluster)
    lables = km.labels_

  return lables







'''
  Funcion para aplicar modelo de clasificación DT, al dataset etiquedado.
  -
'''

def etapaClasfDT(df_tag):
  y = df_tag.Grupo
  X = df_tag.drop(["Grupo","RDT_AJUSTADO"],axis=1)
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234, stratify=y)
  clf = DecisionTreeClassifier()
  clf.fit(X_train,y_train)
  y_pred = clf.predict(X_test)
  accuracy = accuracy_score(y_test, y_pred)
  print(f"Precisión del modelo clasificación: {accuracy * 100:.2f}%")
  return clf



In [14]:
# Lectura del datframe Final
# =================================================================
df = pd.read_csv("../../../Data/Gold/DatasetFinalFP.csv")
df.head(5)

,ID_LOTE,TIPO_SIEMBRA,SEM_TRATADAS,MATERIAL_GENETICO,CULT_ANT,DRENAJE,METODO_COSECHA,ALMACENAMIENTO_FINCA,DIAS_EN_EMERGER,DIAS_EN_EMERGER_A_FLORECER,...,Temp_Max_Avg_Mad,Temp_Min_Avg_Mad,Temp_Avg_Mad,Diurnal_Range_Avg_Mad,Sol_Ener_Accu_Mad,Temp_Max_34_Freq_Mad,Rain_Accu_Mad,Rain_10_Freq_Mad,Rhum_Avg_Mad,RDT_AJUSTADO
0,1910,Manual,NO,Otro,Algodon,SI,Manual,SI,5,46,...,33.41,24.48,28.94,8.94,10617.20,0.36,54.8,0.07,82.21,4576.74
1,1977,Mecanizado,SI,P3966 (Pioneer),Algodon,SI,Manual,SI,5,45,...,33.72,24.51,29.12,9.21,19483.40,0.43,142.1,0.12,81.22,4465.12
2,2006,Mecanizado,NO,P4082 (Pioneer),Algodon,SI,Manual,SI,5,48,...,33.34,24.47,28.91,8.87,12858.10,0.32,88.6,0.12,82.18,4991.86
3,2093,Mecanizado,SI,Otro,Algodon,SI,Manual,SI,5,45,...,33.31,24.41,28.86,8.90,12072.78,0.31,88.6,0.12,82.19,4672.09
4,2096,Mecanizado,NO,Otro,Algodon,SI,Manual,SI,5,46,...,33.28,24.55,28.91,8.73,10746.67,0.34,88.6,0.14,82.31,4576.74


**Nota**
- Se realizarón pruebas variando el tamaño de los cluster [2-10], sin embargo se observo que el numero  de registro para K > 2 era muy pequeño , los grupos formados quedaban por debajo de 40 observaciones, por ese motivo se fijo k=2 y se varia unicamente la semilla de el algortimo de clustering, que permite obtener diferentes distribuciones aleatorias del conjunto de datos

## 1. Prueba Algortimo de Mezclas de Gaussianas

In [27]:
#1. Cargue de los conjuntos de datos
#==============================================
df = pd.read_csv("../../../Data/Gold/DatasetFinalFP.csv")
bin_features =['SEM_TRATADAS','DRENAJE','ALMACENAMIENTO_FINCA','CAP_ENDURE_RASTA','MOTEADOS_RASTA','MOTEADOS_MAS70cm._RASTA',
               'OBSERVA_EROSION_RASTA','OBSERVA_MOHO_RASTA','OBSERVA_RAICES_VIVAS_RASTA','OBSERVA_HOJARASCA_MO_RASTA',
                'SUELO_NEGRO_BLANDO_RASTA','CUCHILLO_PRIMER_HTE_RASTA','CERCA_RIOS_QUEBRADAS_RASTA',
               ]


#2. Codificación de variables categoricas (Codificación dummy)
#   Solo a categoricas
# ============================================================
dataset = OneHotCoding(df,bin_features).dummyCodification()
print("Dimension dataset Original: ", dataset.shape)

#3. Division datos (Variable dependinte e independiente)
# ============================================================
dataset_target = dataset.RDT_AJUSTADO
dataset_features = dataset.drop(["ID_LOTE"], axis=1)

# 4. Normalización del datset
# =================================================================
scaler = MinMaxScaler()
X_normalized = pd.DataFrame(scaler.fit_transform(dataset_features), columns=dataset_features.columns)


# Division de los datos en Training y Test   (PARTE 1)
# ================================================================
dataset_train, dataset_test = train_test_split(X_normalized, test_size = 0.1, random_state=92)
lista_indices_train = list(dataset_train.index)
lista_indices_test = list(dataset_test.index)


dataset_tag_test = dataset_features.loc[lista_indices_test]
y_dataset_test = dataset_tag_test.RDT_AJUSTADO
x_dataset_test = dataset_tag_test.drop(["RDT_AJUSTADO"],axis=1)


# Division de los datos en Training y Test   (PARTE 2)
# ================================================================
d_train , d_test = train_test_split(dataset_train, test_size = 0.1, random_state=87)
lista_indices_train_p1 = list(d_train.index)
lista_indices_test_p1 = list(d_test.index)
d_train_y = d_train.RDT_AJUSTADO
d_train_x = d_train.drop(["RDT_AJUSTADO"],axis=1)

d_test_y = dataset_features.loc[lista_indices_test_p1].RDT_AJUSTADO
d_test_x = dataset_features.loc[lista_indices_test_p1].drop(["RDT_AJUSTADO"],axis=1)


# CLUSTERING (Inicialización de la semilla y mejor r2)
# ==============================================================
semilla = 0
best_r2 = 0

lista_r2 = []
for r in range(30):
  dataset_train_cluster = d_train_x
  gm = GaussianMixture(n_components=2,covariance_type = 'full', random_state=r).fit(dataset_train_cluster)
  target =  gm.predict(dataset_train_cluster)
  dataset_tag = dataset_features.loc[lista_indices_train_p1]
  dataset_tag["Grupo"] =target
  grupos_definitivos  = calcularGruposDefinitivos(dataset_tag)
  print("Grupos Definitivos: ", len(grupos_definitivos))
  # Calculos correlaciones OLMS y modelos
  lista_corr_olsms, models = CalcularCorrelationInitialOLS(grupos_definitivos)
  print("lista Correlaciones Iniciales: ", lista_corr_olsms)
  # Etapa de Clasificación
  model_clasf = etapaClasfDT(dataset_tag)
  # Se valida con lo que nunca se vio.
  grupo_asignado = model_clasf.predict(d_test_x)
  y_pred_test = predictionYield(d_test_x,models,grupo_asignado)
  r2, m_= metricasModelosRegresion(d_test_y, y_pred_test)
  print("r2: ", r2)
  if r2 >= best_r2:
    best_model_clasf = model_clasf
    best_models_reg = models
    best_r2 = r2
    semilla = r


# Con conjunto datos jamas visto  (Conjunto Validación)
# =======================================================
grupo_asignado = best_model_clasf.predict(x_dataset_test)
y_pred_val = predictionYield(x_dataset_test,best_models_reg,grupo_asignado)
r2_new, mdf_= metricasModelosRegresion(y_dataset_test, y_pred_val)


print("-------Mejor R2: ", best_r2)
print("-------Semilla Cluster: ", semilla)
print("-------Dataset de test: r2_new: ", r2_new)

C:\Users\germanm\AppData\Local\Temp\ipykernel_9660\1276057449.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bin_dataset = self.df[self.bin_features].replace({'SI': 1, 'NO': 0})


Ejeción Terminada
Dimension dataset Original:  (799, 176)
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.7431691808719084, 0.8513257852244654]


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.014e+06, tolerance: 8.148e+04
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.577e+07, tolerance: 3.635e+04
  model = cd_fast.enet_coordinate_descent(


Precisión del modelo clasificación: 93.85%
r2:  0.7095804115680177
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.8556693643700475, 0.7417871863817991]


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.566e+07, tolerance: 3.558e+04
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.860e+06, tolerance: 8.203e+04
  model = cd_fast.enet_coordinate_descent(


...

r2:  0.5524569801256453
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.7417871863817991, 0.8556693643700475]


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.860e+06, tolerance: 8.203e+04
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.566e+07, tolerance: 3.558e+04
  model = cd_fast.enet_coordinate_descent(


Precisión del modelo clasificación: 96.92%
r2:  0.652256014276305
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.8513257852244654, 0.7431691808719084]


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.577e+07, tolerance: 3.635e+04
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.014e+06, tolerance: 8.148e+04
  model = cd_fast.enet_coordinate_descent(


Precisión del modelo clasificación: 93.08%
r2:  0.686256958040093
-------Mejor R2:  0.7217598636784883
-------Semilla Cluster:  19
-------Dataset de test: r2_new:  0.7802646350313641


In [28]:
# Obtenemos las metricas del modelo de regresion CLR-2 
# Clustering  GMM
metricasModelosRegresion(y_dataset_test, y_pred_val)

(0.7802646350313641,
   Métrica          Valor
 0     MSE  480734.734147
 1    RMSE     693.350369
 2     MAE     540.901548
 3      R²       0.780265)

## 2. Prueba algortimo de Spectral Clustering

In [18]:
#1. Cargue de los conjuntos de datos
#==============================================
df = pd.read_csv("../../../Data/Gold/DatasetFinalFP.csv")
bin_features =['SEM_TRATADAS','DRENAJE','ALMACENAMIENTO_FINCA','CAP_ENDURE_RASTA','MOTEADOS_RASTA','MOTEADOS_MAS70cm._RASTA',
               'OBSERVA_EROSION_RASTA','OBSERVA_MOHO_RASTA','OBSERVA_RAICES_VIVAS_RASTA','OBSERVA_HOJARASCA_MO_RASTA',
                'SUELO_NEGRO_BLANDO_RASTA','CUCHILLO_PRIMER_HTE_RASTA','CERCA_RIOS_QUEBRADAS_RASTA',
               ]


#2. Codificación de variables categoricas (Codificación dummy)
#   Solo a categoricas
# ============================================================
dataset = OneHotCoding(df,bin_features).dummyCodification()
print("Dimension dataset Original: ", dataset.shape)

#3. Division datos (Variable dependinte e independiente)
# ============================================================
dataset_target = dataset.RDT_AJUSTADO
dataset_features = dataset.drop(["ID_LOTE"], axis=1)

# 4. Normalización del datset
# =================================================================
scaler = MinMaxScaler()
X_normalized = pd.DataFrame(scaler.fit_transform(dataset_features), columns=dataset_features.columns)


# Division de los datos en Training y Test   (PARTE 1)
# ================================================================
dataset_train, dataset_test = train_test_split(X_normalized, test_size = 0.1, random_state=92)
lista_indices_train = list(dataset_train.index)
lista_indices_test = list(dataset_test.index)


dataset_tag_test = dataset_features.loc[lista_indices_test]
y_dataset_test = dataset_tag_test.RDT_AJUSTADO
x_dataset_test = dataset_tag_test.drop(["RDT_AJUSTADO"],axis=1)


# Division de los datos en Training y Test   (PARTE 2)
# ================================================================
d_train , d_test = train_test_split(dataset_train, test_size = 0.1, random_state=87)
lista_indices_train_p1 = list(d_train.index)
lista_indices_test_p1 = list(d_test.index)
d_train_y = d_train.RDT_AJUSTADO
d_train_x = d_train.drop(["RDT_AJUSTADO"],axis=1)

d_test_y = dataset_features.loc[lista_indices_test_p1].RDT_AJUSTADO
d_test_x = dataset_features.loc[lista_indices_test_p1].drop(["RDT_AJUSTADO"],axis=1)


# CLUSTERING
# ==============================================================

n_clusters = 0
semilla = 0
best_r2 = 0


for r in range(31):
  dataset_train_cluster = d_train_x
  sc = SpectralClustering(n_clusters=2,assign_labels='discretize',random_state=r).fit(dataset_train_cluster)
  target = sc.labels_
  dataset_tag = dataset_features.loc[lista_indices_train_p1]
  dataset_tag["Grupo"] =target
  grupos_definitivos  = calcularGruposDefinitivos(dataset_tag)
  print("Grupos Definitivos: ", len(grupos_definitivos))
  # Calculos correlaciones OLMS y modelos
  lista_corr_olsms, models = CalcularCorrelationInitialOLS(grupos_definitivos)
  print("lista Correlaciones Iniciales: ", lista_corr_olsms)
  # Etapa de Clasificación
  model_clasf = etapaClasfDT(dataset_tag)
  # Se valida con lo que nunca se vio.
  grupo_asignado = model_clasf.predict(d_test_x)
  y_pred_test = predictionYield(d_test_x,models,grupo_asignado)
  r2, m_= metricasModelosRegresion(d_test_y, y_pred_test)
  print("r2: ", r2)
  if r2 > best_r2:
    best_model_clasf = model_clasf
    best_models_reg = models
    best_r2 = r2
    semilla = r


# Con conjunto datos jamas visto  (Conjunto Validación)
# =======================================================
grupo_asignado = best_model_clasf.predict(x_dataset_test)
y_pred_val = predictionYield(x_dataset_test,best_models_reg,grupo_asignado)
r2_new, mdf_= metricasModelosRegresion(y_dataset_test, y_pred_val)


print("-------Mejor R2: ", best_r2)
print("-------Semilla Cluster: ", semilla)
print("-------Dataset de test: r2_new: ", r2_new)


C:\Users\germanm\AppData\Local\Temp\ipykernel_9660\1276057449.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bin_dataset = self.df[self.bin_features].replace({'SI': 1, 'NO': 0})


Ejeción Terminada
Dimension dataset Original:  (799, 176)
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.7419823265667131, 0.8577009328173882]


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.160e+06, tolerance: 7.775e+04
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.578e+07, tolerance: 3.952e+04
  model = cd_fast.enet_coordinate_descent(


Precisión del modelo clasificación: 96.15%
r2:  0.7383062047798512
Grupos Definitivos:  2


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.160e+06, tolerance: 7.775e+04
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.578e+07, tolerance: 3.952e+04
  model = cd_fast.enet_coordinate_descent(


...

lista Correlaciones Iniciales:  [0.7419823265667131, 0.8577009328173882]
Precisión del modelo clasificación: 95.38%
r2:  0.7411540368368085
Grupos Definitivos:  2


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.578e+07, tolerance: 3.952e+04
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.160e+06, tolerance: 7.775e+04
  model = cd_fast.enet_coordinate_descent(


lista Correlaciones Iniciales:  [0.8577009328173882, 0.7419823265667131]
Precisión del modelo clasificación: 97.69%
r2:  0.7290110603689446
Grupos Definitivos:  2


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.578e+07, tolerance: 3.952e+04
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.160e+06, tolerance: 7.775e+04
  model = cd_fast.enet_coordinate_descent(


lista Correlaciones Iniciales:  [0.8577009328173882, 0.7419823265667131]
Precisión del modelo clasificación: 96.92%
r2:  0.729060223407625
-------Mejor R2:  0.7411540368368085
-------Semilla Cluster:  3
-------Dataset de test: r2_new:  0.7242583821262185


In [19]:
# Obtenemos las metricas del modelo de regresion CLR-2 
# Clustering  GMM
metricasModelosRegresion(y_dataset_test, y_pred_val)

(0.7242583821262185,
   Métrica          Valor
 0     MSE  603264.628708
 1    RMSE     776.701119
 2     MAE     578.465406
 3      R²       0.724258)

## 3.Prueba Algortimo K-MEANS

In [22]:
#1. Cargue de los conjuntos de datos
#==============================================
df = pd.read_csv("../../../Data/Gold/DatasetFinalFP.csv")
bin_features =['SEM_TRATADAS','DRENAJE','ALMACENAMIENTO_FINCA','CAP_ENDURE_RASTA','MOTEADOS_RASTA','MOTEADOS_MAS70cm._RASTA',
               'OBSERVA_EROSION_RASTA','OBSERVA_MOHO_RASTA','OBSERVA_RAICES_VIVAS_RASTA','OBSERVA_HOJARASCA_MO_RASTA',
                'SUELO_NEGRO_BLANDO_RASTA','CUCHILLO_PRIMER_HTE_RASTA','CERCA_RIOS_QUEBRADAS_RASTA',
               ]


#2. Codificación de variables categoricas (Codificación dummy)
#   Solo a categoricas
# ============================================================
dataset = OneHotCoding(df,bin_features).dummyCodification()
print("Dimension dataset Original: ", dataset.shape)

#3. Division datos (Variable dependinte e independiente)
# ============================================================
dataset_target = dataset.RDT_AJUSTADO
dataset_features = dataset.drop(["ID_LOTE"], axis=1)

# 4. Normalización del datset
# =================================================================
scaler = MinMaxScaler()
X_normalized = pd.DataFrame(scaler.fit_transform(dataset_features), columns=dataset_features.columns)


# Division de los datos en Training y Test   (PARTE 1)
# ================================================================
dataset_train, dataset_test = train_test_split(X_normalized, test_size = 0.1, random_state=92)
lista_indices_train = list(dataset_train.index)
lista_indices_test = list(dataset_test.index)


dataset_tag_test = dataset_features.loc[lista_indices_test]
y_dataset_test = dataset_tag_test.RDT_AJUSTADO
x_dataset_test = dataset_tag_test.drop(["RDT_AJUSTADO"],axis=1)


# Division de los datos en Training y Test   (PARTE 2)
# ================================================================
d_train , d_test = train_test_split(dataset_train, test_size = 0.1, random_state=87)
lista_indices_train_p1 = list(d_train.index)
lista_indices_test_p1 = list(d_test.index)
d_train_y = d_train.RDT_AJUSTADO
d_train_x = d_train.drop(["RDT_AJUSTADO"],axis=1)

d_test_y = dataset_features.loc[lista_indices_test_p1].RDT_AJUSTADO
d_test_x = dataset_features.loc[lista_indices_test_p1].drop(["RDT_AJUSTADO"],axis=1)


# CLUSTERING
# ==============================================================
semilla = 0
best_r2 = 0


for r in range(31):
  dataset_train_cluster = d_train_x

  target = AlgortimoClustering(dataset_train_cluster, 3,r)
  dataset_tag = dataset_features.loc[lista_indices_train_p1]
  dataset_tag["Grupo"] =target
  grupos_definitivos  = calcularGruposDefinitivos(dataset_tag)
  print("Grupos Definitivos: ", len(grupos_definitivos))
  # Calculos correlaciones OLMS y modelos
  lista_corr_olsms, models = CalcularCorrelationInitialOLS(grupos_definitivos)
  print("lista Correlaciones Iniciales: ", lista_corr_olsms)
  # Etapa de Clasificación
  model_clasf = etapaClasfDT(dataset_tag)
  # Se valida con lo que nunca se vio.
  grupo_asignado = model_clasf.predict(d_test_x)
  y_pred_test = predictionYield(d_test_x,models,grupo_asignado)
  r2, m_= metricasModelosRegresion(d_test_y, y_pred_test)
  print("r2: ", r2)
  if r2 > best_r2:
    best_model_clasf = model_clasf
    best_models_reg = models
    best_r2 = r2
    semilla = r

# Con conjunto datos jamas visto  (Conjunto Validación)
# =======================================================
grupo_asignado = best_model_clasf.predict(x_dataset_test)
y_pred_val = predictionYield(x_dataset_test,best_models_reg,grupo_asignado)
r2_new, mdf_= metricasModelosRegresion(y_dataset_test, y_pred_val)


print("-------Mejor R2 Trainig: ", best_r2)
print("-------Semilla Cluster: ", semilla)
print("-------Dataset de test: r2_new: ", r2_new)
  



C:\Users\germanm\AppData\Local\Temp\ipykernel_9660\1276057449.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bin_dataset = self.df[self.bin_features].replace({'SI': 1, 'NO': 0})


Ejeción Terminada
Dimension dataset Original:  (799, 176)
Grupos Definitivos:  2


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.352e+07, tolerance: 8.689e+04
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.444e+07, tolerance: 3.211e+04
  model = cd_fast.enet_coordinate_descent(


lista Correlaciones Iniciales:  [0.7507589252458369, 0.8556769421746891]
Precisión del modelo clasificación: 96.92%
r2:  0.6673996219679856
Grupos Definitivos:  2


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.245e+07, tolerance: 3.208e+04
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.374e+07, tolerance: 8.696e+04
  model = cd_fast.enet_coordinate_descent(


...

lista Correlaciones Iniciales:  [0.8555929156567287, 0.7509033517752091]
Precisión del modelo clasificación: 95.38%
r2:  0.5990101370555683
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.7507589252458369, 0.8556769421746891]


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.352e+07, tolerance: 8.689e+04
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.444e+07, tolerance: 3.211e+04
  model = cd_fast.enet_coordinate_descent(


Precisión del modelo clasificación: 97.69%
r2:  0.6422934796470896
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.7507589252458369, 0.8556769421746891]


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.352e+07, tolerance: 8.689e+04
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.444e+07, tolerance: 3.211e+04
  model = cd_fast.enet_coordinate_descent(


Precisión del modelo clasificación: 96.92%
r2:  0.631597291382072
-------Mejor R2 Trainig:  0.6999563042928757
-------Semilla Cluster:  1
-------Dataset de test: r2_new:  0.7164465468535106


In [23]:
metricasModelosRegresion(y_dataset_test, y_pred_val)

(0.7164465468535106,
   Métrica          Valor
 0     MSE  620355.280245
 1    RMSE     787.626358
 2     MAE     598.584891
 3      R²       0.716447)

### 4. Conclusiones

- Teniendo en cuenta que el mejor desempeño se logro con el algortimo CLR-2 utilizando como algortimo de clustering a mezclas gausianas (GMM),se utilizo este algortimo para definir el algortimo final para predecir el rendimiento del cultivo de maiz y realizar la optimización de practicas agricolas.

- Se exportan lso respectivos modelos (.pkl) y se empaquetan dentro del aplicativo.